# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents

In [ ]:
# Check for the api key

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AIz'):
    print('API key looks good so far')
else:
    print('There might be a problem with your api key')

MODEL = 'gemini-3.1-flash-lite-preview'
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI(base_url=base_url, api_key=api_key)
# alternative to gpt-4.1-mini -> gemini-3-flash


In [ ]:
ed_site = "https://edwarddonner.com"

links = fetch_website_links(ed_site)

links

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company, 
such as links to an about page, or a Company page, or Carrers/Jobs pages.
You should resposne in JSON as in this example:
{
    "links": [
        {"type": "about page", "link": "https://company.com/about"},
        {"type": "contact us page", "link": "https://company.com/contactus"},
        {"type": "careers page", "link": "https://company.com/careers"}
    ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {ed_site} - 
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
print(get_links_user_prompt(ed_site))

In [ ]:
def select_relevant_links(url):    
    messages = [
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(url)}
    ]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        response_format={"type": "json_object"}
    )
    
    result = response.choices[0].message.content
    links = json.loads(result)
    return links


In [ ]:
select_relevant_links(ed_site)